In [ ]:
REPO_URL = "https://github.com/Jaime2pb3/BELEN_LM.git"
BRANCH = "main"
!rm -rf /content/BELEN && git clone --depth 1 --branch $BRANCH $REPO_URL /content/BELEN
%cd /content/BELEN
!git log -1 --format="commit %h · %cd"
!pip -q install sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/content/BELEN'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 51 (delta 0), reused 43 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 438.96 KiB | 16.26 MiB/s, done.
/content/BELEN
commit 0eff0a7 · Mon Nov 23 01:58:55 2026 -0600
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


In [ ]:
REPO_URL = "https://github.com/Jaime2pb3/BELEN_LM.git"
BRANCH = "main"
!rm -rf /content/BELEN && git clone --depth 1 --branch $BRANCH $REPO_URL /content/BELEN
%cd /content/BELEN
!git log -1 --format="commit %h · %cd"
!pip -q install sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/content/BELEN'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 51 (delta 0), reused 43 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 438.96 KiB | 16.26 MiB/s, done.
/content/BELEN
commit 0eff0a7 · Mon Nov 23 01:58:55 2026 -0600
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


# BELEN on Colab
**TwoQuarks Research · Luis Jaime Ledesma Pérez**

Before anything:
#**Runtime → Change runtime type → A100 GPU**.

Each cell is a rung of a ladder.
**If a cell ends in red, do not run the next one.**
Copy the error and fix it there!!

The runner also refuses on its own:
**a stage does not start if the one before it is not green.**

The code always comes from **GitHub**. And Google Drive only keeps data, weights and logs, so a disconnection does not cost you the run.

## 1. Bring the code

In [1]:
REPO_URL = "https://github.com/Jaime2pb3/BELEN_LM.git"
BRANCH = "main"
!rm -rf /content/BELEN && git clone --depth 1 --branch $BRANCH $REPO_URL /content/BELEN
%cd /content/BELEN
!git log -1 --format="commit %h · %cd"
!pip -q install sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/content/BELEN'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 51 (delta 0), reused 43 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 438.96 KiB | 16.26 MiB/s, done.
/content/BELEN
commit 0eff0a7 · Mon Nov 23 01:58:55 2026 -0600
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


## 2. Work folder and model arm

`WORK` holds weights and results, one folder per arm. The data is downloaded once and shared by every arm. Mount Drive if you want all of it to survive a disconnection.

The arm decides which parts of BELEN are on. Change one thing at a time.

| Arm | Phase 3 | Geometric dendrite |
|---|---|---|
| `base` | axon views | off |
| `geometry` | axon views | on |
| `ach` | cholinergic gate | off |
| `ach_geometry` | cholinergic gate | on |

In [2]:
USE_DRIVE = False
ARM = "base"
ITERATIONS = 6500

WORK = "/content/belen_work"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/belen_work"

ARMS = {
    "base": {"AXON_MODE": "views", "GEOMETRIC_DENDRITE": "0"},
    "geometry": {"AXON_MODE": "views", "GEOMETRIC_DENDRITE": "1"},
    "ach": {"AXON_MODE": "ach", "GEOMETRIC_DENDRITE": "0"},
    "ach_geometry": {"AXON_MODE": "ach", "GEOMETRIC_DENDRITE": "1"},
}
import os
os.environ.update(ARMS[ARM])
ENV_FLAGS = " ".join(f"--env {k}={v}" for k, v in ARMS[ARM].items())
DATA_HOME = f"{WORK}/data"
WORK = f"{WORK}/{ARM}"
os.makedirs(DATA_HOME, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{WORK}/data"):
    os.symlink(DATA_HOME, f"{WORK}/data")
print("work :", WORK)
print("data :", DATA_HOME, "(shared by every arm)")
print("arm  :", ENV_FLAGS)

work : /content/belen_work/base
data : /content/belen_work/data (shared by every arm)
arm  : --env AXON_MODE=views --env GEOMETRIC_DENDRITE=0


## 3. The model checks itself (CPU, a few minutes)

The selftest, the regression against the audited base, and the six combinations of Phase 3 and geometry.

In [3]:
!python train_belen.py --selftest
!python tests/test_regression.py
!python tests/test_ach_and_geometry.py
!python train_belen.py --params

belen selftest
  [ok] quarks por token == QuarkBundle original (Δ máx 2.4e-07), y en lote da lo mismo que suelto
  [ok] TopQuark: Gram m×m == covarianza D×D
  [ok] dendrita: concat [(0, 31), (32, 47), (48, 63)] cubre 0..63 sin encimarse · overlap [(0, 31), (0, 15), (0, 15)] (original)
  [ok] axón views: mismo input dos veces Δ 0.0e+00 · cambiar el último token mueve las posiciones previas 0.0e+00 · una secuencia sola == la misma dentro del batch Δ 0.0e+00
  [ok] axón noise (original): mismo input dos veces en eval Δ 1.4e-03 (no determinista)
  [ok] percepción: cerrar el gate 10× reduce lo que entra al Soma a 0.099× sin norm, y a 0.829× con norm (original)
  [ok] LR: constante hasta el paso 4550, lineal a 0 en el 6500 (final 0.000, mitad del warmdown 0.500) · modo reloj explícito baja a 0.00
  [ok] optimizadores: 64 tensores, cada uno en un solo grupo, resid_mix solo en scalar_nowd
  [ok] memoria: coseno medio entre slots +0.25 con sample, +0.76 con promedio de batch + EMA 0.95 (origina

## 4. Cheap rungs: environment, selftests, data, wiring

The 104M model at 6500 iterations × 65536 tokens reads about 426M tokens, that is 5 shards (about 1 GB). There are 195 available.

In [4]:
!python eval_pii/ladder.py env --work $WORK
!python eval_pii/ladder.py selftest --work $WORK
!python eval_pii/ladder.py data --work $WORK --train-shards 5
!python eval_pii/ladder.py wiring --work $WORK $ENV_FLAGS


[escalera] ── 0 · env ────────────────────────────────────────
[escalera] env: VERDE · python 3.13.16 · numpy 2.4.6 · torch 2.14.1+cu130 · GPU NVIDIA A100-SXM4-80GB 79 GB · disco libre 236 GB · 2s

[escalera] ── 1 · selftest ────────────────────────────────────────
leak_forge vforge-0.2 selftest
  [ok] dosis exacta: |visibles| = round(d·n) en 6 tipos × 4 operadores × 7 dosis
  [ok] máscara fiel: '*******.*******ar7@mailbox.test' muestra 14 = 14 visibles, dominio primero
  [ok] blanco d=0: toda posición identificadora queda enmascarada (300 muestras)
  [ok] offsets exactos de spans y señuelos (400 muestras, incluye OCR)
  [ok] determinista con la misma semilla
  [ok] LOD exige monotonía
  [ok] Wilson 19/20 = (0.764, 0.991)
  [ok] regex correo: d=1 sin evasión 1.00, d=1 espaciado 0.17, d=0.5 0.57
  [ok] n-gramas: AUC en prueba = 0.887 (> 0.75)
  [ok] idioma: el inglés no arrastra español, offsets exactos, tasa de positivos es 0.76 / en 0.73
Honestidad: valida el GENERADOR y el EVALUADOR

## 5. Preflight, the first rung that touches the GPU

The real model at real size for 2 iterations, with the real batch and sequence. It catches out of memory, data and checkpoint problems, and builds a smoke card with the real model.

**If it breaks here, it will break worse in the full run.**

### Validation

In [5]:
import sys
sys.path.insert(0, "/content/BELEN")

import torch, torch.nn.functional as F
from belen.quarks import quark_views, TopQuark

torch.manual_seed(0)
phis = [F.layer_norm(torch.randn(2, 16, 768, device="cuda"), (768,)) for _ in range(4)]
legacy = [F.layer_norm(torch.randn(1, 768, device="cuda"), (768,)) for _ in range(4)]

S32, _ = quark_views(phis)
with torch.autocast("cuda", dtype=torch.bfloat16):
    Sbf, _ = quark_views(phis)
    top_bf = TopQuark().compute(legacy)
top_32 = TopQuark().compute(legacy)

print("quark_views fp32 vs autocast  max|Δ|:", (S32 - Sbf).abs().max().item())
print("TopQuark  fp32:", top_32, "| autocast:", top_bf)

quark_views fp32 vs autocast  max|Δ|: 0.0
TopQuark  fp32: 0.35175998982871565 | autocast: 0.0


In [6]:
!python eval_pii/ladder.py preflight --work $WORK $ENV_FLAGS


[escalera] ── 4 · preflight ────────────────────────────────────────
torch.compile: disabled (eager mode)
BELEN  (LEGACY=0)
params: 103,855,023  |  layers: 14  |  dim: 768
heads: 12Q / 12KV  |  mlp_mult: 4
vocab: 1024  |  weight_tying: True  |  RoPE: True
device: cuda:0  |  params float32, cómputo bfloat16 (autocast)  |  sdp: auto
interruptores: dendrite_layout=concat  percept_norm=0  axon_mode=views  resonance_write=sample  eval_isolated=1  ttt_restore=1
batch: 65536 tokens/paso = 8 micro-batches × 8 secuencias × 1024 tokens
DendriticEncoder: field_heads=[8, 4, 4]  kv_heads=4  local=64  mid=256  n_proto=16
ResonanceBuffer: 64 slots × 768d  EMA_decay=0.0  mode=seq  span=8  [no trainable params]
PerceptionAgent: 12 heads  cross-attn(Dendritic × Resonance)
SomaCore: 14 blocks — pure medium, no nucleus
phase3=views  geometric_dendrite=0  n_views=4  λ=0.12
seed: 42  |  world_size: 1  |  grad_accum: 8
optimizadores: emb=786,432  matrix=103,018,496  scalar=23,215  scalar_nowd=26,880
LR: war

## 6. Training

Partial validation every 500 steps, a resumable checkpoint every 250 (if Colab disconnects, run this cell again and it continues) and, at the end, the ablation report: full model, frozen memory, one averaged slot, memory at zero, and no axon or gate.

Test time training only acts during evaluation and is off here, so the saved weights are the trained ones, not the adapted ones.

In [7]:
!python eval_pii/ladder.py train --work $WORK --iterations $ITERATIONS $ENV_FLAGS


[escalera] ── 5 · train ────────────────────────────────────────
  entorno de entrenamiento: TTT_ENABLED=0 VAL_LOSS_EVERY=500 CKPT_EVERY=250 CKPT_DIR=ckpt DATA_PATH=/content/belen_work/base/data/datasets/fineweb10B_sp1024/ TOKENIZER_PATH=/content/belen_work/base/data/tokenizers/fineweb_1024_bpe.model ITERATIONS=6500 RUN_ID=belenc_run AXON_MODE=views GEOMETRIC_DENDRITE=0
torch.compile: disabled (eager mode)
BELEN  (LEGACY=0)
params: 103,855,023  |  layers: 14  |  dim: 768
heads: 12Q / 12KV  |  mlp_mult: 4
vocab: 1024  |  weight_tying: True  |  RoPE: True
device: cuda:0  |  params float32, cómputo bfloat16 (autocast)  |  sdp: auto
interruptores: dendrite_layout=concat  percept_norm=0  axon_mode=views  resonance_write=sample  eval_isolated=1  ttt_restore=1
batch: 65536 tokens/paso = 8 micro-batches × 8 secuencias × 1024 tokens
DendriticEncoder: field_heads=[8, 4, 4]  kv_heads=4  local=64  mid=256  n_proto=16
ResonanceBuffer: 64 slots × 768d  EMA_decay=0.0  mode=seq  span=8  [no trainable

## 7. Preregistration

Criteria, predictions and the hashes of all the code, written **before** measuring. It is written once.

In [8]:
!python eval_pii/ladder.py prereg --work $WORK --langs en
!cat $WORK/ficha/PREREG.md


[escalera] ── 6 · prereg ────────────────────────────────────────
[prereg] escrito /content/belen_work/base/ficha/PREREG.md
[prereg] 3 controles: frente al ruido de inicialización, p >= 0.25
[prereg] Ábrelo, agrega tus predicciones en J1/J2 y después corre `ficha`.
[escalera] prereg: VERDE · escrito ficha/PREREG.md · 1s
# Prerregistro — BELENC-PII sobre BELEN

Escrito: 2026-11-26T23:05:24Z · belen-pii-0.2

## Configuración congelada

```json
{
 "langs": [
  "en"
 ],
 "seed_train": 20261111,
 "seed_val": 20261112,
 "seed_test": 20261113,
 "n_leak": 3000,
 "n_ctx": 500,
 "n_dose": 300,
 "n_groups_val": 90,
 "n_groups_test": 240,
 "lod_reps": 30,
 "probe_seeds": [
  0,
  1,
  2
 ],
 "epochs": 5,
 "random_init_seeds": [
  7,
  8,
  9
 ],
 "memory": "frozen"
}
```

## Criterios

```
Criterios congelados. Se evalúan en PRUEBA, sobre el flujo elegido en VALIDACIÓN.
"Control" = la misma arquitectura con pesos aleatorios. Hay varias inicializaciones y SIEMPRE se compara
contra la más fuerte en

**Your turn:** open `PREREG.md` (files panel, double click), write your predictions in `J1` and `J2`, save. Then measure.

## 8. The card

In [9]:





































!python eval_pii/ladder.py ficha --work $WORK


[escalera] ── 7 · ficha ────────────────────────────────────────
[entrenado/en] [sondas] emb: 3 semillas + control barajado
[entrenado/en] [sondas] dendritic: 3 semillas + control barajado
[entrenado/en] [sondas] percept: 3 semillas + control barajado
[entrenado/en] [sondas] soma0: 3 semillas + control barajado
[entrenado/en] [sondas] soma1: 3 semillas + control barajado
[entrenado/en] [sondas] soma2: 3 semillas + control barajado
[entrenado/en] [sondas] soma3: 3 semillas + control barajado
[entrenado/en] [sondas] soma4: 3 semillas + control barajado
[entrenado/en] [sondas] soma5: 3 semillas + control barajado
[entrenado/en] [sondas] soma6: 3 semillas + control barajado
[entrenado/en] [sondas] soma7: 3 semillas + control barajado
[entrenado/en] [sondas] soma8: 3 semillas + control barajado
[entrenado/en] [sondas] soma9: 3 semillas + control barajado
[entrenado/en] [sondas] soma10: 3 semillas + control barajado
[entrenado/en] [sondas] soma11: 3 semillas + control barajado
/content/BELE

## 8b. Text fingerprint (optional, exploratory, no verdict)

What the model knows about a text reading only 4, 8, 16 or 32 tokens. Language and carrier are trivial and are there so that triviality stays visible. The one that matters is word order on real FineWeb fragments.

In [10]:
!python eval_pii/ladder.py huella --work $WORK


[escalera] ── 8 · huella ────────────────────────────────────────
[huella] idioma: listo
[huella] portadora: listo
[huella] orden: listo
[huella] orden_real: listo
# Huella textual sobre BELEN — huella-0.2

> **EXPLORATORIO.** Sin prerregistro ni veredicto. Números con sus controles, nada más.

Sonda lineal sobre el promedio de los estados de los primeros k tokens. Flujo y reductor (sin o pca64) elegidos en validación. Controles aleatorios: [7, 8, 9]. Fragmento real: fineweb_val_000000.bin.
Modelo: 103,855,023 parámetros · **6500 pasos entrenados** (426M tokens) · ITERATIONS configurado 6500.

## idioma  (azar 0.50, 480 textos de entrenamiento, 240 de prueba, 0 repetidos entre ambos)

Trivial por identidad de token. Si emb y el control ya están arriba, esto no dice nada del modelo.

| tokens leídos | entrenado (IC 95%) | flujo · reductor | emb del entrenado | controles (rango) | ventaja sobre emb (IC) | ventaja sobre el control más fuerte (IC) | barajada: ajuste / prueba |
|---|---|--

## 9. Results

In [11]:
!python eval_pii/ladder.py pack --work $WORK
!python eval_pii/ladder.py status --work $WORK
!ls $WORK


[escalera] ── 9 · pack ────────────────────────────────────────
[escalera] pack: VERDE · belenc_pii_resultados.zip (0.1 MB) · 0s
  [ ok  ] env        python 3.13.16 · numpy 2.4.6 · torch 2.14.1+cu130 · GPU NVIDIA A100-SXM4-80GB 79 GB · disco libre 236 GB
  [ ok  ] selftest   sintaxis de 8 archivos y selftest de 6 módulos
  [ ok  ] data       tokenizer vocab 1024 · 6 fragmentos, 562M tokens, cabeceras verificadas
  [ ok  ] wiring     script real en chico, con ficha de humo
  [ ok  ] preflight  2 iteraciones reales, pesos guardados (417 MB), ficha de humo con el modelo real
  [ ok  ] train      pesos crudos en run_raw.pt (417 MB) · ITERATIONS=6500
  [ ok  ] prereg     escrito ficha/PREREG.md
  [ ok  ] ficha      MODEL_PROBE.md escrito
  [ ok  ] huella     HUELLA.md escrito (exploratorio)
  [ ok  ] pack       belenc_pii_resultados.zip (0.1 MB)
belenc_pii_resultados.zip  huella	   preflight.log     SMOKE_preflight.md
ckpt			   huella.log	   preflight_raw.pt  train.log
data			   ladder.jso

## 10. Paired comparison (optional, long)

The same seed, data and schedule for each arm, one factor at a time. It reuses the data downloaded in step 4. Each arm costs a full training run.

## COMPARE ONLY WHIT SAME GPU (IN THIS CASE: NVIDIA A100-SXM4-40GB, 40960 MiB)

In [12]:
DATA = f"{WORK}/data"
!DATA_PATH=$DATA/datasets/fineweb10B_sp1024/ TOKENIZER_PATH=$DATA/tokenizers/fineweb_1024_bpe.model ITERATIONS=$ITERATIONS VAL_LOSS_EVERY=500 CKPT_DIR=$WORK/ab_ckpt python tools/ab_runs.py --arms base geometry ach ach_geometry --seeds 42

=== ab_base_s42: AXON_MODE=views GEOMETRIC_DENDRITE=0
torch.compile: disabled (eager mode)
BELEN  (LEGACY=0)
params: 103,855,023  |  layers: 14  |  dim: 768
heads: 12Q / 12KV  |  mlp_mult: 4
vocab: 1024  |  weight_tying: True  |  RoPE: True
device: cuda:0  |  params float32, cómputo bfloat16 (autocast)  |  sdp: auto
interruptores: dendrite_layout=concat  percept_norm=0  axon_mode=views  resonance_write=sample  eval_isolated=1  ttt_restore=1
batch: 65536 tokens/paso = 8 micro-batches × 8 secuencias × 1024 tokens
DendriticEncoder: field_heads=[8, 4, 4]  kv_heads=4  local=64  mid=256  n_proto=16
ResonanceBuffer: 64 slots × 768d  EMA_decay=0.0  mode=seq  span=8  [no trainable params]
PerceptionAgent: 12 heads  cross-attn(Dendritic × Resonance)
SomaCore: 14 blocks — pure medium, no nucleus
phase3=views  geometric_dendrite=0  n_views=4  λ=0.12
seed: 42  |  world_size: 1  |  grad_accum: 8
optimizadores: emb=786,432  matrix=103,018,496  scalar=23,215  scalar_nowd=26,880
LR: warmdown lineal en 